<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/StableDiffusionChapter13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# load model
text2img_pipe = StableDiffusionPipeline.from_pretrained(
    "stablediffusionapi/deliberate-v2"
    , torch_dtype = torch.float16
).to("cuda:0")

# generate sample image
prompt = """
high resolution photo,best quality, masterpiece, 8k
A cute cat stand on the tree branch, depth of field, detailed body
"""

neg_prompt = """
paintings,ketches, worst quality, low quality, normal quality, lowres,
monochrome, grayscale
"""

image = text2img_pipe(
    prompt = prompt
    , negative_prompt = neg_prompt
    , generator = torch.Generator("cuda").manual_seed(7)
).images[0]
image

In [ ]:
!pip install opencv-contrib-python
!pip install controlnet_aux

In [ ]:
from controlnet_aux import CannyDetector
canny = CannyDetector()
image_canny = canny(image, 30, 100)

In [ ]:
from diffusers import ControlNetModel
canny_controlnet = ControlNetModel.from_pretrained(
    'takuma104/control_v11'
    , subfolder='control_v11p_sd15_canny'
    , torch_dtype=torch.float16
)

In [ ]:
from diffusers import StableDiffusionControlNetImg2ImgPipeline
cn_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "stablediffusionapi/deliberate-v2"
    , torch_dtype           = torch.float16
    , controlnet            = canny_controlnet
)

In [ ]:
prompt = """
high resolution photo,best quality, masterpiece, 8k
A cute dog stand on the tree branch, depth of field, detailed body
"""

neg_prompt = """
paintings,ketches, worst quality, low quality, normal quality, lowres,
monochrome, grayscale
"""
image_from_canny = single_cn_pipe(
    prompt = prompt
    , negative_prompt = neg_prompt
    , image = canny_image
    , generator = torch.Generator("cuda").manual_seed(2)
    , num_inference_steps = 30
    , guidance_scale = 6.0
).images[0]
image_from_canny

In [ ]:
from controlnet_aux import NormalBaeDetector
normal_bae  = NormalBaeDetector.from_pretrained("lllyasviel/Annotators")
image_canny = normal_bae(image)
image_canny

In [ ]:
from diffusers import ControlNetModel
canny_controlnet = ControlNetModel.from_pretrained(
    'takuma104/control_v11'
    , subfolder='control_v11p_sd15_canny'
    , torch_dtype=torch.float16
)
bae_controlnet = ControlNetModel.from_pretrained(
    'takuma104/control_v11'
    , subfolder='control_v11p_sd15_normalbae'
    , torch_dtype=torch.float16
)
controlnets = [canny_controlnet, bae_controlnet]

In [ ]:
from diffusers import StableDiffusionControlNetPipeline
two_cn_pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stablediffusionapi/deliberate-v2"
    , torch_dtype           = torch.float16
    , controlnet            = controlnets
).to("cuda")

In [ ]:
prompt = """
high resolution photo,best quality, masterpiece, 8k
A cute dog on the tree branch, depth of field, detailed body,
"""

neg_prompt = """
paintings,ketches, worst quality, low quality, normal quality, lowres,
monochrome, grayscale
"""
image_from_2cn = two_cn_pipe(
    prompt = prompt
    , image                 = [canny_image,bae_image]
    , controlnet_conditioning_scale = [0.5,0.5]
    , generator             = torch.Generator("cuda").manual_seed(2)
    , num_inference_steps   = 30
    , guidance_scale        = 5.5
).images[0]
image_from_2cn

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
sdxl_pipe = StableDiffusionXLPipeline.from_pretrained(
    "RunDiffusion/RunDiffusion-XL-Beta"
    , torch_dtype           = torch.float16
    , load_safety_checker   = False
)
sdxl_pipe.watermark = None

In [ ]:
from diffusers import EulerDiscreteScheduler
prompt = """
full body photo of young man, arms spread
white blank background,
glamour photography,
upper body wears shirt,
wears suit pants,
wears leather shoes
"""
neg_prompt = """
worst quality,low quality, paint, cg, spots, bad hands,
three hands, noise, blur, bad anatomy, low resolution, blur face, bad face
"""
sdxl_pipe.to("cuda")

sdxl_pipe.scheduler = EulerDiscreteScheduler.from_config(sdxl_pipe.scheduler.config)
image = sdxl_pipe(
    prompt              = prompt
    , negative_prompt   = neg_prompt
    , width             = 832
    , height            = 1216
).images[0]
sdxl_pipe.to("cpu")
torch.cuda.empty_cache()
image

In [ ]:
from controlnet_aux import OpenposeDetector
open_pose = OpenposeDetector.from_pretrained("lllyasviel/Annotators")
pose = open_pose(image)
pose

In [ ]:
from diffusers import StableDiffusionXLControlNetPipeline
from diffusers import ControlNetModel
sdxl_pose_controlnet = ControlNetModel.from_pretrained(
    "thibaud/controlnet-openpose-sdxl-1.0"
    , torch_dtype=torch.float16
)

sdxl_cn_pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "RunDiffusion/RunDiffusion-XL-Beta"
    , torch_dtype           = torch.float16
    , load_safety_checker   = False
    , add_watermarker       = False
    , controlnet            = sdxl_pose_controlnet
)
sdxl_cn_pipe.watermark = None

In [ ]:
from diffusers import EulerDiscreteScheduler
prompt = """
full body photo of young woman, arms spread
white blank background,
glamour photography,
wear sunglass,
upper body wears shirt,
wears suit pants,
wears leather shoes
"""
neg_prompt = """
worst quality,low quality, paint, cg, spots, bad hands,
three hands, noise, blur, bad anatomy, low resolution,
blur face, bad face
"""
sdxl_cn_pipe.to("cuda")

sdxl_cn_pipe.scheduler = EulerDiscreteScheduler.from_config(sdxl_cn_pipe.scheduler.config)
generator = torch.Generator("cuda").manual_seed(2)

image = sdxl_cn_pipe(
    prompt              = prompt
    , negative_prompt   = neg_prompt
    , width             = 832
    , height            = 1216
    , image             = pose
    , generator         = generator
    , controlnet_conditioning_scale = 0.5
    , num_inference_steps = 30
    , guidance_scale    = 6.0
).images[0]
sdxl_cn_pipe.to("cpu")
torch.cuda.empty_cache()
image